# gen_optuna_v6 — Targeting > 0.88232 LB
### Key upgrades over all previous versions
- **CatBoost native categorical handling** (biggest untried lever — no ordinal encoding for CatBoost)
- **LGBM DART boosting** mode added to search space
- **Smoothed target encoding** (Bayesian smoothing, avoids leakage on rare categories)
- **Seed averaging** (3 seeds × 10 folds, proven in v5)
- **Logistic Regression meta** (more stable than LGBM meta on 5 features)
- **Wider Optuna search** (bagging_temperature up to 2.0, DART drop_rate)
- OOF → LB gap is ~0.008; need OOF ≥ 0.876 to beat 0.88232

In [1]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm catboost imbalanced-learn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 0. Imports & Config

In [2]:
import pandas as pd
import numpy as np
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
N_TRIALS    = 120   # per model
CV_SPLITS   = 5     # inner Optuna folds
N_SPLITS    = 10    # outer OOF folds
SEEDS       = [42, 7, 123]  # multi-seed averaging (proven in v5)

/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [3]:
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

print('Train:', TRAIN_DATA.shape, '| Test:', TEST_DATA.shape)

Train: (29839, 16) | Test: (19893, 16)


## 2. Feature Engineering

In [4]:
def add_features(df):
    df = df.copy()

    # ── Campaign history ──────────────────────────────────────
    df['prev_success']        = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted']     = (df['previous'] == 0).astype(int)
    df['pdays_clean']         = df['pdays'].apply(lambda x: 999 if x == -1 else x)
    df['was_contacted_before']= (df['pdays'] != -1).astype(int)
    df['prev_contacts_log']   = np.log1p(df['previous'])
    df['pdays_log']           = np.log1p(df['pdays_clean'])

    # ── Call duration ─────────────────────────────────────────
    df['duration_log']        = np.log1p(df['duration'])
    df['long_call']           = (df['duration'] > 300).astype(int)
    df['very_long_call']      = (df['duration'] > 600).astype(int)
    df['no_duration']         = (df['duration'] == 0).astype(int)

    # ── Balance ───────────────────────────────────────────────
    df['log_balance']         = np.log1p(df['balance'].clip(lower=0))
    df['is_debt']             = (df['balance'] < 0).astype(int)
    df['balance_bin']         = pd.cut(
        df['balance'].clip(-5000, 50000),
        bins=[-np.inf, 0, 500, 2000, 8000, np.inf], labels=[0,1,2,3,4]
    ).astype(float)

    # ── Campaign ──────────────────────────────────────────────
    df['log_campaign']        = np.log1p(df['campaign'])
    df['over_contacted']      = (df['campaign'] > 5).astype(int)

    # ── Time ──────────────────────────────────────────────────
    df['month_sin']           = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']           = np.cos(2 * np.pi * df['month'] / 12)
    if 'day' in df.columns:
        df['day_sin']         = np.sin(2 * np.pi * df['day'] / 31)
        df['day_cos']         = np.cos(2 * np.pi * df['day'] / 31)
        df['end_of_month']    = (df['day'] >= 25).astype(int)

    # ── Age ───────────────────────────────────────────────────
    df['age_young']           = (df['age'] < 30).astype(int)
    df['age_senior']          = (df['age'] >= 55).astype(int)

    # ── Interactions ──────────────────────────────────────────
    df['long_call_x_success'] = df['long_call'] * df['prev_success']
    df['duration_x_prev']     = df['duration_log'] * df['prev_contacts_log']
    df['duration_x_success']  = df['duration_log'] * df['prev_success']
    df['pdays_x_prev']        = (1 / (df['pdays_clean'] + 1)) * df['prev_contacts_log']
    df['balance_x_nodebt']    = df['log_balance'] * (1 - df['is_debt'])
    df['duration_per_contact']= df['duration_log'] / (df['log_campaign'] + 1)
    df['recent_success']      = df['prev_success'] * (df['pdays_clean'] < 90).astype(int)

    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)
print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA  shape:', TEST_DATA.shape)

TRAIN_DATA shape: (29839, 45)
TEST_DATA  shape: (19893, 45)


## 3. Encoding

> **Key design**: we create TWO versions of X_train/X_test:
> - `X_train_te` — ordinal-encoded + smoothed target encoding → for XGB, LGBM, ET
> - `X_train_cat` — raw strings kept for cat columns → for CatBoost native handling

In [5]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

# ── For XGB / LGBM / ET: ordinal encode cats ─────────────────
ENCODER = OrdinalEncoder(
    handle_unknown='use_encoded_value', unknown_value=-1
).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index
    )
    return pd.concat([cat_enc, df[num_cols].copy()], axis=1)

X_train_ord = preprocess(TRAIN_DATA, ENCODER)
X_test_ord  = preprocess(TEST_DATA,  ENCODER)
y_train     = TRAIN_LABEL['subscription'].values

# ── Smoothed target encoding (Bayesian, avoids leakage on rare cats) ──
def target_encode_cv_smooth(X_tr, y_tr, X_te, cols, n_splits=5, smooth=20):
    """CV target encoding with smoothing: (n*cat_mean + smooth*global_mean)/(n+smooth)"""
    skf         = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc    = X_tr.copy()
    X_te_enc    = X_te.copy()
    global_mean = y_tr.mean()
    for col in cols:
        oof     = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))
        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            grp    = pd.DataFrame({'cat': X_tr[col].iloc[fold_tr_idx].values,
                                   'y':   y_tr.iloc[fold_tr_idx].values})
            stats  = grp.groupby('cat')['y'].agg(['sum','count'])
            smoothed = (stats['sum'] + smooth * global_mean) / (stats['count'] + smooth)
            oof[fold_val_idx] = (
                X_tr[col].iloc[fold_val_idx].map(smoothed).fillna(global_mean).values
            )
            te_vals += (
                X_te[col].reset_index(drop=True).map(smoothed).fillna(global_mean).values
                / n_splits
            )
        X_tr_enc[col + '_te'] = oof
        X_te_enc[col + '_te'] = te_vals
    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train_ord.index)
X_train_te, X_test_te = target_encode_cv_smooth(X_train_ord, y_series, X_test_ord, cat_cols)
print('X_train_te shape:', X_train_te.shape)

# ── For CatBoost: keep raw string cats, fill NaN ─────────────
def preprocess_cat(df):
    df = df.copy()
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna('MISSING')
    return df

X_train_cat = preprocess_cat(TRAIN_DATA)
X_test_cat  = preprocess_cat(TEST_DATA)
CAT_FEATURE_INDICES = [X_train_cat.columns.get_loc(c) for c in cat_cols]
print('CatBoost cat feature indices:', CAT_FEATURE_INDICES)

X_train_te shape: (29839, 53)
CatBoost cat feature indices: [1, 2, 3, 4, 6, 7, 8, 15]


## 4. Helpers

In [6]:
from sklearn.metrics import balanced_accuracy_score, roc_curve
from sklearn.ensemble import ExtraTreesClassifier
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

X_arr      = X_train_te.values
X_te_arr   = X_test_te.values
X_cat_arr  = X_train_cat.values
X_te_cat   = X_test_cat.values
scale_pos  = (y_train == 0).sum() / (y_train == 1).sum()
inner_skf  = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)

def youden_ba_cv(model, X, y, cv, cat_features=None):
    """OOF Youden-J balanced accuracy. Pass cat_features for CatBoost."""
    oof = np.zeros(len(y))
    for tr_idx, val_idx in cv.split(X, y):
        if cat_features is not None:
            model.fit(X[tr_idx], y[tr_idx], cat_features=cat_features)
        else:
            model.fit(X[tr_idx], y[tr_idx])
        oof[val_idx] = model.predict_proba(X[val_idx])[:, 1]
    fpr, tpr, ths = roc_curve(y, oof)
    best_t = float(ths[np.argmax(tpr - fpr)])
    return balanced_accuracy_score(y, (oof >= best_t).astype(int))

## 5. Optuna Tuning

In [7]:
# ── XGBoost ───────────────────────────────────────────────────
def xgb_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 300, 2000),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 10),
        min_child_weight = trial.suggest_int('min_child_weight', 3, 100),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        colsample_bylevel= trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-3, 20.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
        gamma            = trial.suggest_float('gamma', 0.0, 5.0),
        scale_pos_weight = scale_pos, eval_metric='logloss',
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    return youden_ba_cv(XGBClassifier(**params), X_arr, y_train, inner_skf)

print('Tuning XGBoost...')
xgb_study = optuna.create_study(direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_xgb = xgb_study.best_params
print(f'  Best XGB BA: {xgb_study.best_value:.5f}')
print(f'  Params: {best_xgb}')

Tuning XGBoost...


Best trial: 99. Best value: 0.873158: 100%|██████████| 120/120 [38:18<00:00, 19.15s/it]

  Best XGB BA: 0.87316
  Params: {'n_estimators': 1682, 'learning_rate': 0.005835151069731103, 'max_depth': 10, 'min_child_weight': 5, 'subsample': 0.8168444554101115, 'colsample_bytree': 0.42393842192938797, 'colsample_bylevel': 0.95424369175353, 'reg_alpha': 12.583903273158358, 'reg_lambda': 0.005045757885965306, 'gamma': 3.672343356652346}


In [8]:
# ── LightGBM  (gbdt + dart both in search) ────────────────────
def lgbm_objective(trial):
    boosting = trial.suggest_categorical('boosting_type', ['gbdt', 'dart'])
    params = dict(
        boosting_type    = boosting,
        n_estimators     = trial.suggest_int('n_estimators', 300, 2000),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 12),
        num_leaves       = trial.suggest_int('num_leaves', 15, 200),
        min_child_samples= trial.suggest_int('min_child_samples', 5, 150),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-3, 20.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
    )
    if boosting == 'dart':
        params['drop_rate'] = trial.suggest_float('drop_rate', 0.05, 0.3)
    return youden_ba_cv(LGBMClassifier(**params), X_arr, y_train, inner_skf)

print('Tuning LightGBM (gbdt + dart)...')
lgbm_study = optuna.create_study(direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_lgbm = lgbm_study.best_params
print(f'  Best LGBM BA: {lgbm_study.best_value:.5f}')
print(f'  Params: {best_lgbm}')

Tuning LightGBM (gbdt + dart)...


Best trial: 83. Best value: 0.874134:  99%|█████████▉| 119/120 [6:39:26<03:21, 201.40s/it]  


[W 2026-04-06 20:41:25,293] Trial 119 failed with parameters: {'boosting_type': 'dart', 'n_estimators': 1445, 'learning_rate': 0.0509196058536513, 'max_depth': 7, 'num_leaves': 161, 'min_child_samples': 96, 'subsample': 0.9012208530129499, 'colsample_bytree': 0.988120831570914, 'reg_alpha': 10.678588289081667, 'reg_lambda': 0.002899033814866281, 'drop_rate': 0.1984058845454359} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/w7/h8ptyfn56xnfdf1y_gqj5ssw0000gn/T/ipykernel_3845/1068935797.py", line 19, in lgbm_objective
    return youden_ba_cv(LGBMClassifier(**params), X_arr, y_train, inner_skf)
  File "/var/folders/w7/h8ptyfn56xnfdf1y_gqj5ssw0000gn/T/ipykernel_3845/3289307659.py", line 21, in youden_ba_cv
    model.fit(X[tr_idx], y[tr_idx])
    ~~~~~

KeyboardInterrupt: 

In [ ]:
# ── CatBoost  (NATIVE cat features — untried in all previous versions!) ──
def cat_objective(trial):
    params = dict(
        iterations          = trial.suggest_int('iterations', 300, 2500),
        learning_rate       = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        depth               = trial.suggest_int('depth', 3, 10),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 20.0),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 2.0),  # wider!
        random_strength     = trial.suggest_float('random_strength', 0.0, 5.0),
        min_data_in_leaf    = trial.suggest_int('min_data_in_leaf', 1, 50),
        auto_class_weights  = 'Balanced',
        eval_metric         = 'Logloss',
        random_seed         = RANDOM_SEED,
        verbose             = 0,
    )
    model = CatBoostClassifier(**params)
    # Use native cat features on raw string data
    return youden_ba_cv(model, X_cat_arr, y_train, inner_skf,
                        cat_features=CAT_FEATURE_INDICES)

print('Tuning CatBoost (native cats)...')
cat_study = optuna.create_study(direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_cat = cat_study.best_params
print(f'  Best CAT BA: {cat_study.best_value:.5f}')
print(f'  Params: {best_cat}')

In [ ]:
# ── ExtraTrees (diverse, decorrelated from gradient boosters) ─
from sklearn.ensemble import ExtraTreesClassifier

def et_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 300, 1500),
        max_depth        = trial.suggest_int('max_depth', 5, 30),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 3, 80),
        max_features     = trial.suggest_float('max_features', 0.3, 1.0),
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1,
    )
    return youden_ba_cv(ExtraTreesClassifier(**params), X_arr, y_train, inner_skf)

print('Tuning ExtraTrees...')
et_study = optuna.create_study(direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
et_study.optimize(et_objective, n_trials=60, show_progress_bar=True)
best_et = et_study.best_params
print(f'  Best ET BA: {et_study.best_value:.5f}')

print('\n' + '='*50)
print('OPTUNA SUMMARY')
print('='*50)
for name, study in [('XGB',xgb_study),('LGBM',lgbm_study),('CAT',cat_study),('ET',et_study)]:
    print(f'  {name:5s}: {study.best_value:.5f}')

## 6. Multi-Seed OOF Generation (10-fold × 3 seeds)

In [ ]:
model_names = ['XGB', 'LGBM', 'CAT', 'ET']
oof_preds   = {n: np.zeros(len(y_train))   for n in model_names}
test_preds  = {n: np.zeros(len(X_test_te)) for n in model_names}

for seed in SEEDS:
    print(f'\n=== Seed {seed} ===')
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    oof_s  = {n: np.zeros(len(y_train))   for n in model_names}
    test_s = {n: np.zeros(len(X_test_te)) for n in model_names}

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
        X_tr,    X_val    = X_arr[tr_idx],    X_arr[val_idx]
        X_tr_c,  X_val_c  = X_cat_arr[tr_idx],X_cat_arr[val_idx]
        y_tr,    y_val    = y_train[tr_idx],  y_train[val_idx]

        # XGB
        m = XGBClassifier(**best_xgb, scale_pos_weight=scale_pos,
                          eval_metric='logloss', random_state=seed, n_jobs=-1)
        m.fit(X_tr, y_tr)
        oof_s['XGB'][val_idx]  = m.predict_proba(X_val)[:, 1]
        test_s['XGB']         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        # LGBM
        m = LGBMClassifier(**best_lgbm, class_weight='balanced',
                           random_state=seed, n_jobs=-1, verbose=-1)
        m.fit(X_tr, y_tr)
        oof_s['LGBM'][val_idx] = m.predict_proba(X_val)[:, 1]
        test_s['LGBM']        += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        # CatBoost — native cats, uses X_cat_arr !
        m = CatBoostClassifier(**best_cat, auto_class_weights='Balanced',
                               eval_metric='Logloss', random_seed=seed, verbose=0)
        m.fit(X_tr_c, y_tr, cat_features=CAT_FEATURE_INDICES)
        oof_s['CAT'][val_idx]  = m.predict_proba(X_val_c)[:, 1]
        test_s['CAT']         += m.predict_proba(X_te_cat)[:, 1] / N_SPLITS

        # ExtraTrees
        m = ExtraTreesClassifier(**best_et, class_weight='balanced',
                                  random_state=seed, n_jobs=-1)
        m.fit(X_tr, y_tr)
        oof_s['ET'][val_idx]   = m.predict_proba(X_val)[:, 1]
        test_s['ET']          += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        print(f'  Fold {fold+1}/{N_SPLITS} done')

    for n in model_names:
        oof_preds[n]  += oof_s[n]  / len(SEEDS)
        test_preds[n] += test_s[n] / len(SEEDS)

print('\nOOF Balanced Accuracy per model (Youden J):')
oof_ba_scores = {}
for name in model_names:
    fpr, tpr, ths = roc_curve(y_train, oof_preds[name])
    best_t = float(ths[np.argmax(tpr - fpr)])
    ba = balanced_accuracy_score(y_train, (oof_preds[name] >= best_t).astype(int))
    oof_ba_scores[name] = ba
    print(f'  {name:5s}: BA={ba:.5f}  threshold={best_t:.4f}')

## 7. Ensemble — LR Meta + Weighted Average

In [ ]:
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression

def rank_norm(arr):
    return rankdata(arr) / len(arr)

oof_rank  = np.column_stack([rank_norm(oof_preds[n])  for n in model_names])
test_rank = np.column_stack([rank_norm(test_preds[n]) for n in model_names])

meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)

# ── Meta A: Logistic Regression (stable, low variance) ───────
oof_lr  = np.zeros(len(y_train))
test_lr = np.zeros(len(X_te_arr))
for tr_idx, val_idx in meta_skf.split(oof_rank, y_train):
    lr = LogisticRegression(C=1.0, max_iter=2000, random_state=42)
    lr.fit(oof_rank[tr_idx], y_train[tr_idx])
    oof_lr[val_idx] = lr.predict_proba(oof_rank[val_idx])[:, 1]
final_lr = LogisticRegression(C=1.0, max_iter=2000, random_state=42)
final_lr.fit(oof_rank, y_train)
test_lr = final_lr.predict_proba(test_rank)[:, 1]
print('LR meta coefficients:', dict(zip(model_names, final_lr.coef_[0].round(3))))

# ── BA-weighted average ───────────────────────────────────────
weights = np.array([oof_ba_scores[n] for n in model_names])
weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-9)
weights /= weights.sum()
print('\nBA-proportional weights:')
for n, w in zip(model_names, weights):
    print(f'  {n}: {w:.4f}')
oof_wavg  = oof_rank  @ weights
test_wavg = test_rank @ weights

# ── Grid-search best blend (LR meta vs weighted avg) ─────────
best_ba_blend, best_w = 0.0, 0.6
for w in np.arange(0.2, 0.9, 0.02):
    oof_b = w * oof_lr + (1 - w) * oof_wavg
    fpr, tpr, ths = roc_curve(y_train, oof_b)
    bt = float(ths[np.argmax(tpr - fpr)])
    ba = balanced_accuracy_score(y_train, (oof_b >= bt).astype(int))
    if ba > best_ba_blend:
        best_ba_blend, best_w = ba, w

print(f'\nBest blend weight for LR meta: {best_w:.2f}')

oof_blend  = best_w * oof_lr  + (1 - best_w) * oof_wavg
test_blend = best_w * test_lr + (1 - best_w) * test_wavg

# ── Youden-J threshold ────────────────────────────────────────
fpr_b, tpr_b, thresholds_b = roc_curve(y_train, oof_blend)
best_threshold = float(thresholds_b[np.argmax(tpr_b - fpr_b)])
best_ba = balanced_accuracy_score(y_train, (oof_blend >= best_threshold).astype(int))

print(f'\nBlended OOF BA   : {best_ba:.5f}')
print(f'Optimal threshold: {best_threshold:.4f}')
print(f'Expected LB ~    : {best_ba + 0.0084:.5f}')

# Fine sweep
print('\nThreshold | #Pred-1 | OOF BA')
print('-'*40)
for t in np.arange(max(0.01, best_threshold-0.05), min(0.99, best_threshold+0.06), 0.002):
    preds = (oof_blend >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    mark  = ' <- best' if abs(t - best_threshold) < 0.002 else ''
    print(f'  {t:.3f} | {preds.sum():6d} | {ba:.5f}{mark}')

## 8. Decision Gate + Submit

In [ ]:
SUBMIT_GATE = 0.876  # need OOF >= 0.876 to expect LB > 0.88232

if best_ba >= SUBMIT_GATE:
    print(f'✅ OOF {best_ba:.5f} >= {SUBMIT_GATE} — SUBMIT')
else:
    print(f'❌ OOF {best_ba:.5f} < {SUBMIT_GATE} — investigate before submitting')

test_classes = (test_blend >= best_threshold).astype(int)
n1, n0 = test_classes.sum(), (test_classes == 0).sum()
print(f'\nPrediction dist — 0: {n0}, 1: {n1}  (pos-rate {n1/(n0+n1)*100:.1f}%)')
print(f'Blended OOF BA : {best_ba:.5f}')
print(f'Threshold used : {best_threshold:.4f}')

submission = pd.DataFrame({'id': TEST_DATA.index, 'subscription': test_classes})
submission.to_csv('submission_v6.csv', index=False)
print('\nSaved submission_v6.csv ✓')
print(submission['subscription'].value_counts())